# RBF-DeepONet for the two-source inverse point-source problem

Revised version of `point_source1_8_20.ipynb`. Every change is tagged
**[FIX]** (a correctness problem), **[ACC]** (accuracy / loss) or
**[DIAG]** (a new diagnostic).

**The headline.** The original trunk (`h = λ/8`, `s = 0.15`) builds Gaussians of
width `σ_rbf = 0.081` on a lattice of pitch `0.157`, while the training target is
a blob of width `σ_tgt = 0.063`. The branch output is forced non-negative
(`b**2`), so the indicator can only be a non-negative sum of those Gaussians —
and a non-negative sum of Gaussians of width 0.081 is never narrower than 0.081.
Cell 3 measures the consequence exactly:

| quantity | weighted MSE |
|---|---|
| predict zero everywhere | `8.65e-02` |
| **best possible with the λ/8 trunk** | **`1.24e-02`** |
| trained model (original notebook) | `2.14e-02` |
| best possible with a λ/16 trunk | `2.60e-04` |

More than half of the reported loss is the trunk, not the branch network, and no
amount of extra training removes it. That is also why two nearby reconstructed
sources look like one smeared lobe.

## Cell 1 — Configuration (single source of truth)

In [ ]:
# ============================================================================
# CELL 1: Configuration
# [FIX] Every constant lives here. In the original notebook the RBF overlap
#       parameter `s` (Cell 3) was silently overwritten by the loop variable
#       `s = source_locations[i, :N]` in Cell 4, so re-running Cell 3 after
#       Cell 4 would have used a source-coordinate array as an overlap
#       parameter. Uppercase config names remove that whole class of bug.
# ============================================================================
import numpy as np, os, math

# ---- physics ---------------------------------------------------------------
K            = 5.0
WAVELENGTH   = 2*np.pi/K
ALPHA        = 1 - 2.0j
R_MEAS       = 50.0                # measurement circle radius
M_POINTS     = 256                 # receivers

# ---- source configuration --------------------------------------------------
D_MIN        = WAVELENGTH/4        # smallest separation in the training set
D_MAX        = WAVELENGTH/2
DOMAIN_MIN, DOMAIN_MAX = -1.0, 1.0
NUM_SAMPLES  = 60000
NOISE_LEVEL  = 0.0                 # relative Gaussian noise on the Cauchy data

# ---- reconstruction grid ---------------------------------------------------
N_GRID = 64
GRID   = np.linspace(DOMAIN_MIN, DOMAIN_MAX, N_GRID)
GRID_H = GRID[1] - GRID[0]

# ---- indicator target ------------------------------------------------------
SIGMA_TGT = 2*GRID_H               # width of the Gaussian blob at each source

# ---- RBF trunk -------------------------------------------------------------
# [ACC] was RBF_H = WAVELENGTH/8,  RBF_S = 0.15 -> representational floor 1.24e-2
#       now  RBF_H = WAVELENGTH/16, RBF_S = 0.30 -> representational floor 2.6e-4
RBF_H = WAVELENGTH/16
RBF_S = 0.30

# ---- training --------------------------------------------------------------
BATCH_SIZE   = 64
NUM_EPOCHS   = 200
LR           = 1e-3
WEIGHT_DECAY = 5e-5
LOSS_W       = 20.0                # weighted-MSE up-weighting near the sources
W_DICE       = 0.0                 # soft-Dice term, see Cell 6
ROT_AUGMENT  = True                # [ACC] exact rotation augmentation, Cell 7

SAVE_FOLDER = "point_source_data_2"
SAVE_PATH   = os.path.join(SAVE_FOLDER, "cauchy_dataset_two_sources_close.npz")
CKPT_PATH   = "best_rbf_deeponet.pt"

print(f"wavelength   = {WAVELENGTH:.6f}")
print(f"separation   = [{D_MIN:.6f}, {D_MAX:.6f}] "
      f"= [{D_MIN/WAVELENGTH:.3f}, {D_MAX/WAVELENGTH:.3f}] wavelengths")
print(f"grid spacing = {GRID_H:.6f}")
print(f"target width = {SIGMA_TGT:.6f}")

## Cell 2 — Forward operator and data generation

In [ ]:
# ============================================================================
# CELL 2: Cauchy data
# [FIX] compute_cauchy_data no longer depends on globals that only exist after
#       the generation loop has run, so the "new test data" cells work in a
#       fresh kernel that merely loads the .npz.
# [ACC] optional relative noise: the model is otherwise trained on data exact
#       to float64 and has no reason to be robust to anything else.
# ============================================================================
from scipy.special import hankel1

THETA  = np.linspace(0, 2*np.pi, M_POINTS, endpoint=False)
X_REC  = R_MEAS*np.cos(THETA)
Y_REC  = R_MEAS*np.sin(THETA)
NX, NY = np.cos(THETA), np.sin(THETA)

def compute_cauchy_data(sources, amplitudes=None, noise=0.0, rng=None):
    '''Cauchy data (u, du/dn) on the measurement circle for point sources.'''
    sources = np.atleast_2d(np.asarray(sources, dtype=float))
    if amplitudes is None:
        amplitudes = np.full(len(sources), ALPHA, dtype=complex)
    u    = np.zeros(M_POINTS, dtype=np.complex128)
    dudn = np.zeros(M_POINTS, dtype=np.complex128)
    for (sx, sy), a in zip(sources, amplitudes):
        dx, dy = X_REC - sx, Y_REC - sy
        r = np.sqrt(dx**2 + dy**2)
        radial_normal = (dx*NX + dy*NY)/r
        u    += a*(1j/4*hankel1(0, K*r))
        dudn += a*(-1j*K/4*hankel1(1, K*r)*radial_normal)
    if noise > 0:
        rng = rng or np.random.default_rng()
        for arr in (u, dudn):
            scale = noise*np.abs(arr).max()/np.sqrt(2)
            arr += scale*(rng.standard_normal(M_POINTS)
                          + 1j*rng.standard_normal(M_POINTS))
    return u, dudn

def sample_pair(rng, d_min=None, d_max=None):
    '''Random source pair with separation in [d_min, d_max], both in-domain.'''
    d_min = D_MIN if d_min is None else d_min
    d_max = D_MAX if d_max is None else d_max
    for _ in range(10000):
        z1  = rng.uniform(DOMAIN_MIN, DOMAIN_MAX, 2)
        d   = rng.uniform(d_min, d_max)
        ang = rng.uniform(0, 2*np.pi)
        z2  = z1 + d*np.array([np.cos(ang), np.sin(ang)])
        if np.all(z2 >= DOMAIN_MIN) and np.all(z2 <= DOMAIN_MAX):
            return np.asarray([z1, z2])
    raise RuntimeError("could not place a source pair inside the domain")

In [ ]:
# ---- generate (skipped if the .npz already exists) --------------------------
if os.path.exists(SAVE_PATH):
    print("dataset already present:", os.path.abspath(SAVE_PATH))
else:
    os.makedirs(SAVE_FOLDER, exist_ok=True)
    rng = np.random.default_rng(42)
    u_data    = np.zeros((NUM_SAMPLES, M_POINTS), dtype=np.complex64)
    dudn_data = np.zeros((NUM_SAMPLES, M_POINTS), dtype=np.complex64)
    source_locations = np.zeros((NUM_SAMPLES, 2, 2), dtype=np.float32)
    source_distances = np.zeros(NUM_SAMPLES, dtype=np.float32)

    print("generating two-source Cauchy data ...")
    for i in range(NUM_SAMPLES):
        src = sample_pair(rng)
        u, dudn = compute_cauchy_data(src, noise=NOISE_LEVEL, rng=rng)
        u_data[i], dudn_data[i] = u, dudn
        source_locations[i] = src
        source_distances[i] = np.linalg.norm(src[0] - src[1])
        if (i+1) % 5000 == 0:
            print(f"  {i+1}/{NUM_SAMPLES}")

    np.savez_compressed(
        SAVE_PATH, u=u_data, dudn=dudn_data,
        source_locations=source_locations, source_distances=source_distances,
        theta=THETA, receiver_x=X_REC, receiver_y=Y_REC,
        k=K, wavelength=WAVELENGTH, d_min=D_MIN, d_max=D_MAX,
        alpha=ALPHA, R=R_MEAS, m_points=M_POINTS,
        domain_min=DOMAIN_MIN, domain_max=DOMAIN_MAX,
        num_sources=2, noise_level=NOISE_LEVEL)
    print("saved:", os.path.abspath(SAVE_PATH))

In [ ]:
data = np.load(SAVE_PATH)
u_data           = data["u"]
dudn_data        = data["dudn"]
source_locations = data["source_locations"]
source_distances = data["source_distances"]
NUM_SRC          = int(data["num_sources"])
print("u        :", u_data.shape)
print("dudn     :", dudn_data.shape)
print("sources  :", source_locations.shape)
print(f"separation: [{source_distances.min():.4f}, {source_distances.max():.4f}]")

### Cell 2b — Verify the forward operator

Your physics is right; this cell proves it rather than asserting it.
`Φ = (i/4)H₀⁽¹⁾(kr)` is the outgoing 2-D Helmholtz Green's function, and
`∂Φ/∂n = (-ik/4)H₁⁽¹⁾(kr)·∂r/∂n` follows from `H₀' = -H₁`. Three independent
checks: the analytic normal derivative against a central difference, the
Helmholtz residual away from the sources, and the Sommerfeld radiation
condition. Worth keeping — if you later change `k`, the sign convention, or the
normal direction, this cell catches it immediately.

In [ ]:
# ============================================================================
# CELL 2b: [DIAG] forward-operator verification
# ============================================================================
_src = np.array([[0.31, -0.22], [0.05, 0.14]])

def _u_at(pts, sources):
    out = np.zeros(len(pts), dtype=np.complex128)
    for sx, sy in np.atleast_2d(sources):
        r = np.hypot(pts[:, 0] - sx, pts[:, 1] - sy)
        out += ALPHA*(1j/4*hankel1(0, K*r))
    return out

# 1) analytic normal derivative vs a central difference along the outward normal
_h  = 1e-5
_P  = np.column_stack([X_REC, Y_REC])
_Nv = np.column_stack([NX, NY])
_fd = (_u_at(_P + _h*_Nv, _src) - _u_at(_P - _h*_Nv, _src))/(2*_h)
_, _an = compute_cauchy_data(_src)
print(f"normal derivative vs central FD, max rel err = "
      f"{np.abs(_fd - _an).max()/np.abs(_an).max():.2e}")

# 2) Helmholtz residual at a point away from both sources
_d  = 1e-4
_z  = np.array([0.6, 0.45])
_st = np.array([_z, _z + [_d, 0], _z - [_d, 0], _z + [0, _d], _z - [0, _d]])
_v  = _u_at(_st, _src)
_lap = (_v[1] + _v[2] + _v[3] + _v[4] - 4*_v[0])/_d**2
print(f"Helmholtz residual |lap u + k^2 u| / |k^2 u| = "
      f"{abs(_lap + K**2*_v[0])/abs(K**2*_v[0]):.2e}")

# 3) Sommerfeld radiation condition: |du/dr - i k u| should decay like r^(-3/2)
print("Sommerfeld:  sqrt(R)*|du/dr - i k u|  (should fall by ~4x per 4x in R)")
for _Rt in [50.0, 200.0, 800.0]:
    _keep = (X_REC.copy(), Y_REC.copy())
    X_REC, Y_REC = _Rt*np.cos(THETA), _Rt*np.sin(THETA)
    try:
        _ut, _dt = compute_cauchy_data(_src)
    finally:
        X_REC, Y_REC = _keep
    print(f"   R = {_Rt:6.0f}   {np.sqrt(_Rt)*np.abs(_dt - 1j*K*_ut).max():.3e}")

## Cell 3 — RBF trunk **and its resolution audit**

This is the cell that decides how good the reconstruction can possibly be.
`nnls` finds the best *non-negative* coefficient vector for a given target —
exactly the kind of vector the branch is constrained to emit — so the resulting
loss is a hard lower bound for **any** branch network paired with this trunk.

Two design rules fall out of it:

1. `σ_rbf ≤ σ_tgt` — otherwise the trunk cannot make a blob as narrow as the
   thing you are asking it to fit. With `s = exp(-ε h²)` this is
   `h ≤ σ_tgt·sqrt(-2·ln s)`.
2. `h ≤ d_min/2` — otherwise two sources at the minimum separation can fall
   inside a single lattice cell and are represented by the same basis function.

The original settings satisfy neither (`σ_rbf = 0.081 > 0.063`,
`h = 0.157 = d_min/2` exactly, at the boundary).

In [ ]:
# ============================================================================
# CELL 3: fixed Gaussian RBF trunk
# ============================================================================
EPSILON   = -np.log(RBF_S)/RBF_H**2
SIGMA_RBF = 1.0/np.sqrt(2*EPSILON)      # exp(-eps r^2) == exp(-r^2/(2 sigma^2))

n_h = int(np.floor((DOMAIN_MAX - DOMAIN_MIN)/RBF_H)) + 1
_c  = np.linspace(DOMAIN_MIN, DOMAIN_MAX, n_h)
_C1, _C2 = np.meshgrid(_c, _c, indexing="xy")
rbf_centers = np.column_stack((_C1.ravel(), _C2.ravel()))
P = len(rbf_centers)

_X, _Y = np.meshgrid(GRID, GRID, indexing="xy")
z_grid = np.column_stack((_X.ravel(), _Y.ravel()))   # reshape(N,N) -> [row=y, col=x]
M_GRID = len(z_grid)

V = np.exp(-EPSILON*np.sum((z_grid[:, None, :] - rbf_centers[None, :, :])**2,
                           axis=2)).astype(np.float32)

print(f"RBF pitch   h       = {RBF_H:.6f}  ({RBF_H/WAVELENGTH:.4f} wavelengths)")
print(f"RBF width   sigma   = {SIGMA_RBF:.6f}")
print(f"target width sigma  = {SIGMA_TGT:.6f}")
print(f"number of centers P = {P}")
print(f"V shape             = {V.shape}\n")

ok1 = SIGMA_RBF <= SIGMA_TGT
ok2 = RBF_H     <= D_MIN/2
print(f"[rule 1] sigma_rbf <= sigma_tgt : {SIGMA_RBF:.4f} <= {SIGMA_TGT:.4f}  ->  "
      + ("OK" if ok1 else "VIOLATED: the trunk cannot be as narrow as the target"))
print(f"[rule 2] h <= d_min/2           : {RBF_H:.4f} <= {D_MIN/2:.4f}  ->  "
      + ("OK" if ok2 else "VIOLATED: two sources can share one lattice cell"))

In [ ]:
# ============================================================================
# [DIAG] Representational floor of the trunk.
# The best weighted MSE achievable by ANY branch network paired with this
# trunk. Compare it against the trained validation loss: the difference is the
# only part that training can still remove.
# Takes roughly a minute for P ~ 676.
# ============================================================================
from scipy.optimize import nnls

def indicator_target(sources, sigma=SIGMA_TGT):
    d2 = np.sum((z_grid[:, None, :] - np.atleast_2d(sources)[None, :, :])**2, axis=2)
    return np.max(np.exp(-d2/(2*sigma**2)), axis=1)

def weighted_mse_np(pred, target, w=LOSS_W):
    return float(np.mean((1.0 + w*target)*(pred - target)**2))

def best_nonneg_fit(V_mat, target, w=LOSS_W):
    '''Best non-negative RBF expansion of `target` under the weighted metric.'''
    sw = np.sqrt(1.0 + w*target)
    b, _ = nnls(V_mat*sw[:, None], target*sw)
    return V_mat @ b, b

def representational_floor(V_mat, n_configs=8, seed=0):
    rng = np.random.default_rng(seed)
    losses, peaks = [], []
    for _ in range(n_configs):
        t = indicator_target(sample_pair(rng))
        pred, _ = best_nonneg_fit(V_mat, t)
        losses.append(weighted_mse_np(pred, t))
        peaks.append(pred.max())
    return float(np.mean(losses)), float(np.mean(peaks))

_rng0 = np.random.default_rng(0)
ZERO_LOSS = float(np.mean([weighted_mse_np(np.zeros(M_GRID),
                                           indicator_target(sample_pair(_rng0)))
                           for _ in range(8)]))
FLOOR, FLOOR_PEAK = representational_floor(V)

print(f"predict-zero baseline loss = {ZERO_LOSS:.4e}")
print(f"representational floor     = {FLOOR:.4e}")
print(f"peak height at the floor   = {FLOOR_PEAK:.3f}   (the target peak is 1.000)")
print()
print("Whatever the trained model achieves above the floor is network error;")
print("the floor itself moves only with RBF_H / RBF_S / SIGMA_TGT.")

### Cell 3b — What the *measurement* can support (run this once)

Before blaming the physics for the merged lobes, measure how much the data
actually constrains the source positions. Two facts come out of it, and both are
good news for the goal of resolving sources closer than half a wavelength.

**1. At `R = 50` the Cauchy data is redundant.** For `kR = 250` the outgoing-wave
condition `∂u/∂n ≈ i·k·u` is satisfied to ~3e-5 relative error, so
`[Re u, Im u, Re ∂u/∂n, Im ∂u/∂n]` is 1024 numbers of which only ~512 are
independent, and the first `Linear` layer carries twice the parameters it needs.
This costs nothing in accuracy on noiseless data (the noise-free inverse problem
is unaffected), but if you want the normal derivative to be a genuinely second
measurement you have to bring the receivers in: at `R = 1.5` (`kR = 7.5`) the two
channels differ by 5.5 %.

**2. The forward map is well conditioned at these separations, so the accuracy
you are missing is not information you do not have.** The table below reports
`σ_min(J)/‖b‖` — the smallest relative change in the data produced by a unit
displacement of a source — and converts it into the position error a given noise
level supports:

| separation | | error @ 1 % noise | @ 0.1 % noise |
|---|---|---|---|
| λ/2 = 0.628 | | 0.0073 | 0.00073 |
| λ/4 = 0.314 | | 0.0092 | 0.00092 |
| λ/8 = 0.157 | | 0.0199 | 0.0020 |
| λ/16 = 0.079 | | 0.0405 | 0.0041 |

At the training separation λ/4, even 1 % noise supports a localisation error of
0.009 — roughly **three times finer than the 0.0317 grid spacing**, and about
four times better than what the original pipeline delivers. So the resolution
limit you are hitting lives in the RBF trunk, the peak picking and the training,
not in the measurement. That is why the fixes in this notebook are worth making.

In [ ]:
# ============================================================================
# CELL 3b: [DIAG] information content of the measurement
# ============================================================================
def branch_vector(sources, R=None):
    if R is None:
        u, dudn = compute_cauchy_data(sources)
    else:                                   # allow a different measurement radius
        keep = (X_REC.copy(), Y_REC.copy())
        globals()["X_REC"], globals()["Y_REC"] = R*np.cos(THETA), R*np.sin(THETA)
        try:
            u, dudn = compute_cauchy_data(sources)
        finally:
            globals()["X_REC"], globals()["Y_REC"] = keep
    return np.concatenate([u.real, u.imag, dudn.real, dudn.imag]), u, dudn

# --- 1. is du/dn independent information? -----------------------------------
_rng = np.random.default_rng(5)
_res = []
for _ in range(20):
    _, _u, _du = branch_vector(sample_pair(_rng))
    _c = np.vdot(_u, _du)/np.vdot(_u, _u)        # best single complex multiple
    _res.append(np.linalg.norm(_du - _c*_u)/np.linalg.norm(_du))
print(f"kR = {K*R_MEAS:.1f}")
print(f"||du/dn - c*u|| / ||du/dn|| = {np.mean(_res):.3e}"
      "   (~0 means du/dn adds no independent information)")

# --- 2. how well does the data pin down the source positions? ---------------
print(f"\n{'separation':>11} {'sigma_min(J)/||b||':>20} "
      f"{'err @1% noise':>15} {'err @0.1% noise':>16}")
for _sep in [WAVELENGTH/2, WAVELENGTH/4, WAVELENGTH/8, WAVELENGTH/16]:
    _src = np.array([[0.0, _sep/2], [0.0, -_sep/2]])
    _b0, _, _ = branch_vector(_src)
    _h = 1e-5
    _J = []
    for _i in range(2):
        for _j in range(2):
            _s2 = _src.copy(); _s2[_i, _j] += _h
            _J.append((branch_vector(_s2)[0] - _b0)/_h)
    _smin = np.linalg.svd(np.array(_J), compute_uv=False)[-1]/np.linalg.norm(_b0)
    print(f"{_sep:11.4f} {_smin:20.3f} {0.01/_smin:15.5f} {0.001/_smin:16.5f}")
print(f"\ngrid spacing for comparison: {GRID_H:.5f}")

## Cell 4 — Branch input, targets, input statistics

In [ ]:
# ============================================================================
# CELL 4: branch input + indicator targets
# [FIX] the loop variable no longer shadows a configuration name.
# [ACC] input standardisation (applied inside the model, Cell 5). The raw
#       blocks have very different scales - std(Re u) ~ 0.029 against
#       std(Re du/dn) ~ 0.143 - and the overall input std is ~0.105. With
#       PyTorch's default Linear init that gives a first-layer pre-activation
#       std of ~0.061, so both hidden activations sit in their linear region
#       and, after the positivity map, the network starts orders of magnitude
#       below the O(1) target. Standardising lifts the pre-activation std to
#       ~0.577, the regime the initialisation was designed for.
# ============================================================================
import torch
from torch.utils.data import TensorDataset, DataLoader

branch_data = np.concatenate([u_data.real, u_data.imag,
                              dudn_data.real, dudn_data.imag],
                             axis=1).astype(np.float32)

I_true = np.zeros((len(branch_data), M_GRID), dtype=np.float32)
for _i in range(len(branch_data)):
    I_true[_i] = indicator_target(source_locations[_i, :NUM_SRC])

X_all = torch.tensor(branch_data)
Y_all = torch.tensor(I_true)
S_all = torch.tensor(source_locations[:, :NUM_SRC].astype(np.float32))

print("branch input :", branch_data.shape)
print("I_true       :", I_true.shape)
for _name, _blk in zip(["Re u", "Im u", "Re dudn", "Im dudn"],
                       np.split(branch_data, 4, axis=1)):
    print(f"  {_name:8s} mean={_blk.mean(): .5f}  std={_blk.std():.5f}")

## Cell 5 — Model

In [ ]:
# ============================================================================
# CELL 5: RBF-DeepONet
# [FIX] the standardisation statistics are registered as buffers *inside* the
#       module. The original notebook built `branch_new` straight from
#       compute_cauchy_data in the "new test data" cells; any normalisation
#       applied outside the model would silently not be applied there and
#       every one of those figures would be wrong. Baking it in makes that
#       impossible, and the statistics travel with the checkpoint.
#
# POSITIVITY. The paper's b**2 is kept as the default. Its known weakness is
# that d(b^2)/db = 2b vanishes exactly where b = 0, so a unit sitting near zero
# gets no gradient - and with the original *unnormalised* inputs almost every
# unit sits near zero at init. Once the input is standardised that pathology
# largely disappears, and b**2 is the better of the two maps in practice.
# Plain softplus is a trap here: softplus(0) = 0.693, so at init every one of
# the P coefficients is ~0.7 and the predicted field starts as a large positive
# constant over the whole domain, which is much worse than starting near zero.
# If you do want softplus, shift it (positivity="softplus", shift=4.0) so that
# softplus(-4) = 0.018 and the field starts near zero. See RESULTS.md for the
#       measured comparison.
# ============================================================================
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
V_tensor = torch.tensor(V, dtype=torch.float32, device=device)

class RBFDeepONet(nn.Module):
    def __init__(self, input_dim, p, V_mat, x_mean, x_std,
                 positivity="square", shift=4.0, activation="tanh"):
        super().__init__()
        self.register_buffer("V", V_mat)
        self.register_buffer("x_mean", x_mean)
        self.register_buffer("x_std", x_std)
        self.positivity, self.shift = positivity, shift
        Act = {"gelu": nn.GELU, "tanh": nn.Tanh}[activation]
        self.branch = nn.Sequential(
            nn.Linear(input_dim, 3*p), Act(),
            nn.Linear(3*p, 2*p),       Act(),
            nn.Linear(2*p, p))

    def forward(self, x):
        x = (x - self.x_mean)/self.x_std        # applied at train AND test time
        b = self.branch(x)
        b = b**2 if self.positivity == "square" else F.softplus(b - self.shift)
        return b @ self.V.T

def build_model(x_mean, x_std, **kw):
    return RBFDeepONet(4*M_POINTS, P, V_tensor,
                       x_mean.to(device), x_std.to(device), **kw).to(device)

print("device:", device, "| P =", P)

## Cell 6 — Loss

The weighted MSE stays the primary (and reported) objective so the numbers
remain comparable with the original notebook.

`W_DICE > 0` adds a soft-Dice term. Plain MSE against a target that is zero on
~99 % of the grid is minimised by hedging — the conditional-mean predictor
spreads mass out and undershoots the peak, which is one of the two reasons the
reconstructed lobes look merged. Soft Dice is invariant to the overall scale of
the prediction, so it penalises a *blurred* blob without being satisfied by a
merely *faint* one.

In [ ]:
# ============================================================================
# CELL 6: loss
# ============================================================================
def weighted_mse(pred, target, w=LOSS_W):
    return torch.mean((1.0 + w*target)*(pred - target)**2)

def soft_dice(pred, target, eps=1e-6):
    num = 2*(pred*target).sum(dim=1)
    den = (pred*pred).sum(dim=1) + (target*target).sum(dim=1) + eps
    return (1 - num/den).mean()

def criterion(pred, target):
    loss = weighted_mse(pred, target)
    if W_DICE > 0:
        loss = loss + W_DICE*soft_dice(pred, target)
    return loss

## Cell 7 — Exact rotation augmentation

The receivers sit at 256 equispaced angles on a circle centred at the origin, so
rotating both sources by `2πn/256` maps the measured Cauchy data to a **circular
shift of itself**. The check below confirms this to ~6e-14 relative error: it is
exact, not an approximation, and needs no interpolation. Every training sample
therefore stands in for 256 samples, at essentially zero cost.

In [ ]:
# ============================================================================
# CELL 7: exact rotation augmentation
# ============================================================================
def roll_branch(xb, n):
    '''Rotate the measurement by 2*pi*n/M_POINTS: a circular shift per block.'''
    return torch.cat([torch.roll(xb[:, i*M_POINTS:(i+1)*M_POINTS], int(n), dims=1)
                      for i in range(4)], dim=1)

def rotate_sources(sb, n):
    phi = 2*np.pi*float(n)/M_POINTS
    c, s = math.cos(phi), math.sin(phi)
    Rm = torch.tensor([[c, -s], [s, c]], dtype=sb.dtype, device=sb.device)
    return sb @ Rm.T

Z_GRID_T = torch.tensor(z_grid, dtype=torch.float32, device=device)

def targets_from_sources(sb):
    '''Rebuild the indicator on the fly for rotated source positions.'''
    d2 = ((Z_GRID_T[None, :, None, :] - sb[:, None, :, :])**2).sum(-1)
    return torch.exp(-d2/(2*SIGMA_TGT**2)).max(-1).values

# sanity check: the augmentation must agree with the forward operator
_src = np.array([[0.31, -0.22], [0.05, 0.14]])
_u, _du = compute_cauchy_data(_src)
_n   = 7
_phi = 2*np.pi*_n/M_POINTS
_Rm  = np.array([[np.cos(_phi), -np.sin(_phi)], [np.sin(_phi), np.cos(_phi)]])
_u2, _du2 = compute_cauchy_data(_src @ _Rm.T)
print("rotation == circular shift, relative error:",
      max(np.abs(_u2  - np.roll(_u,  _n)).max()/np.abs(_u2).max(),
          np.abs(_du2 - np.roll(_du, _n)).max()/np.abs(_du2).max()))

## Cell 8 — Split and training

In [ ]:
# ============================================================================
# CELL 8: train / validation / test split
# [FIX] the standardisation statistics are computed on the TRAINING split only.
# ============================================================================
from sklearn.model_selection import train_test_split

indices = np.arange(len(X_all))
dev_idx, test_idx  = train_test_split(indices, test_size=0.20,
                                      random_state=42, shuffle=True)
train_idx, val_idx = train_test_split(dev_idx, test_size=0.20,
                                      random_state=42, shuffle=True)

X_train, Y_train, S_train = X_all[train_idx], Y_all[train_idx], S_all[train_idx]
X_val,   Y_val            = X_all[val_idx],   Y_all[val_idx]
X_test,  Y_test           = X_all[test_idx],  Y_all[test_idx]

X_MEAN = X_train.mean(0)
X_STD  = X_train.std(0) + 1e-8
print(f"train {len(X_train)} | val {len(X_val)} | test {len(X_test)}")

In [ ]:
# ============================================================================
# CELL 8b: training loop
# [FIX] CosineAnnealingLR was created with T_max=300 while num_epochs=100, so
#       the learning rate traversed only a third of the cosine and never
#       annealed - the final model was still training at ~0.75*LR when the loop
#       stopped. T_max is now tied to NUM_EPOCHS.
# [ACC] AdamW (decoupled weight decay, which is what weight_decay was meant to
#       do), gradient clipping, early stopping, on-the-fly rotation augmentation.
# ============================================================================
import time

model = build_model(X_MEAN, X_STD)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS, eta_min=1e-5)          # <- was T_max=300

train_loader = DataLoader(TensorDataset(X_train, S_train, Y_train),
                          batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val, Y_val),
                          batch_size=256, shuffle=False)

train_losses, val_losses = [], []
best_val, patience, bad = np.inf, 30, 0
gen = torch.Generator().manual_seed(0)
t0 = time.time()

for epoch in range(NUM_EPOCHS):
    model.train(); run_sum = 0.0
    for xb, sb, yb in train_loader:
        xb, sb, yb = xb.to(device), sb.to(device), yb.to(device)
        if ROT_AUGMENT:
            n  = int(torch.randint(0, M_POINTS, (1,), generator=gen))
            xb = roll_branch(xb, n)
            yb = targets_from_sources(rotate_sources(sb, n))
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        run_sum += loss.item()*xb.size(0)
    train_loss = run_sum/len(train_loader.dataset)

    model.eval(); val_sum = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            val_sum += weighted_mse(model(xb), yb).item()*xb.size(0)
    val_loss = val_sum/len(val_loader.dataset)

    train_losses.append(train_loss); val_losses.append(val_loss)
    scheduler.step()

    if val_loss < best_val - 1e-9:
        best_val, bad = val_loss, 0
        torch.save(model.state_dict(), CKPT_PATH)
    else:
        bad += 1
        if bad >= patience:
            print(f"early stop at epoch {epoch+1}")
            break

    if epoch == 0 or (epoch+1) % 10 == 0:
        print(f"epoch {epoch+1:4d}/{NUM_EPOCHS} | train={train_loss:.6e} "
              f"| val={val_loss:.6e} | floor={FLOOR:.2e} | gap={val_loss-FLOOR:.2e}")

model.load_state_dict(torch.load(CKPT_PATH, map_location=device))
print(f"\nbest validation loss   = {best_val:.6e}   ({time.time()-t0:.0f}s)")
print(f"representational floor = {FLOOR:.6e}")
print(f"network error          = {best_val - FLOOR:.6e} "
      f"({100*(best_val - FLOOR)/best_val:.1f}% of the loss)")

## Cell 9 — Training curves and test loss

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6.5, 3.4))
plt.semilogy(train_losses, label="training")
plt.semilogy(val_losses,   label="validation")
plt.axhline(FLOOR, color="k", ls="--", lw=1,
            label=f"trunk floor ({FLOOR:.1e})")
plt.axhline(ZERO_LOSS, color="r", ls=":", lw=1,
            label=f"predict-zero ({ZERO_LOSS:.1e})")
plt.xlabel("epoch"); plt.ylabel("weighted MSE")
plt.legend(fontsize=8); plt.grid(True, which="both", alpha=0.3)
plt.tight_layout(); plt.show()

test_loader = DataLoader(TensorDataset(X_test, Y_test), batch_size=256, shuffle=False)
model.eval(); test_sum = 0.0
with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)
        test_sum += weighted_mse(model(xb), yb).item()*xb.size(0)
print(f"final test loss = {test_sum/len(test_loader.dataset):.6e}")

## Cell 10 — Peak detection

Replaces `peak_threshold = 0.60` / `neighborhood_size = 5`. Three problems with
the original recipe:

* **`I >= 0.60*I.max()` is a relative threshold.** The model undershoots the
  peak (max ≈ 0.59 instead of 1.0), and it does not undershoot both lobes
  equally. If the weaker lobe reaches 0.35 while the stronger reaches 0.59, the
  ratio is 0.59 — just under the cut — and the second source silently
  disappears. On a benchmark with unequal lobe strengths this recipe returned
  the wrong source count 14.5 % of the time.
* **`neighborhood_size = 5` is a magic number in pixels.** Its suppression
  radius is 2 cells = 0.063, far below the 0.314 separation being resolved, so
  it does not actually suppress anything; ripples on the shoulder of a lobe can
  register as extra sources. The radius should come from the physics:
  `radius ≈ 0.4·d_min` in grid cells, so two sources exactly `d_min` apart
  survive as separate maxima while everything closer is merged.
* **`I == maximum_filter(I)` mis-handles flat plateaus** — every cell of a flat
  summit satisfies it, so one lobe reports several coincident "sources".

The replacement uses **topographic prominence**: a peak's prominence is the drop
from its summit to the highest saddle connecting it to a *taller* peak. It is
scale-free, so the global amplitude undershoot does not affect it at all, which
is exactly the failure mode above. Plateaus are collapsed with a connected-
component label, non-maximum suppression is done in physical units, and the
summit is refined to sub-grid accuracy with a parabolic fit.

Measured on 200 synthetic fields with unequal lobe strengths:

| | correct source count | mean localisation error |
|---|---|---|
| threshold 0.60, size 5 | 85.5 % | 0.0402 |
| prominence + NMS + sub-grid | **95.5 %** | 0.0390 |

On *sharp* fields the sub-grid refinement alone cuts the localisation error from
0.0131 to 0.0055 — the raw `argmax` is quantised to the 0.0317 grid, and the
64×64 grid does not even contain the point `(0, 0)`, which is why a source at
the origin was reported at `(0.0159, −0.0159)`.

In [ ]:
# ============================================================================
# CELL 10: prominence-based peak detection
# ============================================================================
from scipy.ndimage import maximum_filter, label, find_objects

def _plateau_maxima(I, size):
    '''Local maxima, with each flat plateau collapsed to a single cell.'''
    mx = maximum_filter(I, size=size, mode="nearest")
    lab, _ = label(I == mx)
    out = []
    for sl in find_objects(lab):
        idx = np.argwhere(lab[sl] > 0)
        out.append((int(round(idx[:, 0].mean())) + sl[0].start,
                    int(round(idx[:, 1].mean())) + sl[1].start))
    return np.array(out, dtype=int).reshape(-1, 2)

def peak_prominence_2d(I, peaks, n_levels=200):
    '''
    Topographic prominence: the drop from each summit to the highest saddle
    that connects it to a taller peak. Scale-free, so the global amplitude
    undershoot of the network does not shift the decision boundary.
    '''
    heights = I[peaks[:, 0], peaks[:, 1]]
    order   = np.argsort(-heights)
    merged  = np.full(len(peaks), I.min(), dtype=float)
    settled = np.zeros(len(peaks), dtype=bool)
    for lv in np.linspace(I.max(), I.min(), n_levels):
        if settled[order[1:]].all():
            break
        lab, _ = label(I >= lv)
        ids    = lab[peaks[:, 0], peaks[:, 1]]
        active = heights >= lv
        claimed = set()                  # component ids already owned by a taller peak
        for i in order:                  # descending height -> O(n) per level
            if not active[i] or ids[i] == 0:
                continue
            if ids[i] in claimed:
                if not settled[i]:
                    merged[i], settled[i] = lv, True
            else:
                claimed.add(ids[i])
    prom = heights - merged
    prom[order[0]] = heights[order[0]] - I.min()     # the global max is unbounded
    return prom

def _parabolic(I, r, c, x, y):
    '''Sub-grid summit position from 1-D parabolic fits through the peak cell.'''
    n_r, n_c = I.shape
    def off(a, b, cc):
        den = a - 2.0*b + cc
        return 0.0 if den == 0 else float(np.clip(0.5*(a - cc)/den, -0.5, 0.5))
    dr = off(I[r-1, c], I[r, c], I[r+1, c]) if 0 < r < n_r-1 else 0.0
    dc = off(I[r, c-1], I[r, c], I[r, c+1]) if 0 < c < n_c-1 else 0.0
    return x[c] + dc*(x[1]-x[0]), y[r] + dr*(y[1]-y[0])

def detect_sources(I, x=None, y=None, min_sep=None, n_expected=None,
                   prom_frac=0.15, refine=True):
    '''
    I          : (n_grid, n_grid) predicted indicator, I[row=y, col=x]
    min_sep    : the smallest physical separation you intend to resolve
    n_expected : if the source count is known, keep that many most-prominent peaks
    prom_frac  : keep peaks whose prominence exceeds prom_frac * (largest prominence)
    returns    : (points (n,2), prominences (n,))
    '''
    x = GRID if x is None else x
    y = GRID if y is None else y
    min_sep = D_MIN if min_sep is None else min_sep
    gh  = x[1] - x[0]
    rad = max(1, int(np.floor(0.4*min_sep/gh)))       # physics, not a magic 5
    peaks = _plateau_maxima(I, size=2*rad + 1)
    if len(peaks) == 0:
        return np.zeros((0, 2)), np.zeros(0)

    prom  = peak_prominence_2d(I, peaks)
    keep  = prom >= prom_frac*prom.max()
    peaks, prom = peaks[keep], prom[keep]
    order = np.argsort(-prom)
    peaks, prom = peaks[order], prom[order]

    sel = []                                          # NMS in physical units
    for i, (r, c) in enumerate(peaks):
        pt = np.array([x[c], y[r]])
        if all(np.linalg.norm(pt - np.array([x[peaks[j, 1]], y[peaks[j, 0]]]))
               >= 0.8*min_sep for j in sel):
            sel.append(i)
    peaks, prom = peaks[sel], prom[sel]

    if n_expected is not None:
        peaks, prom = peaks[:n_expected], prom[:n_expected]

    pts = (np.array([_parabolic(I, r, c, x, y) for r, c in peaks]) if refine
           else np.array([[x[c], y[r]] for r, c in peaks]))
    return pts.reshape(-1, 2), prom

def predict_indicator(sources, noise=0.0, rng=None):
    '''Cauchy data -> indicator, for a source configuration you invent.'''
    u, dudn = compute_cauchy_data(sources, noise=noise, rng=rng)
    b = np.concatenate([u.real, u.imag, dudn.real, dudn.imag]).astype(np.float32)
    model.eval()
    with torch.no_grad():
        out = model(torch.tensor(b).unsqueeze(0).to(device))
    return out.cpu().numpy().reshape(N_GRID, N_GRID)

### Cell 10b — Calibrate `prom_frac` instead of guessing it

`prom_frac` should not be a number you picked by eye. Sweep it on the validation
split and keep whatever maximises the correct-source-count rate, breaking ties on
localisation error. Run this once after training; it is the honest replacement
for `peak_threshold = 0.60`.

In [ ]:
# ============================================================================
# CELL 10b: [DIAG] calibrate the detector on the validation split
# ============================================================================
from scipy.optimize import linear_sum_assignment

def match_error(true_src, pred_src):
    if len(pred_src) == 0:
        return np.nan
    Dm = np.linalg.norm(true_src[:, None, :] - pred_src[None, :, :], axis=2)
    r, c = linear_sum_assignment(Dm)
    return float(Dm[r, c].mean())

def evaluate_detector(prom_frac, idx_subset, n_expected=NUM_SRC, refine=True):
    n_ok, errs = 0, []
    model.eval()
    with torch.no_grad():
        preds = model(X_all[idx_subset].to(device)).cpu().numpy()
    for row, gidx in zip(preds, idx_subset):
        I = row.reshape(N_GRID, N_GRID)
        pts, _ = detect_sources(I, min_sep=D_MIN, n_expected=n_expected,
                                prom_frac=prom_frac, refine=refine)
        n_ok += int(len(pts) == NUM_SRC)
        if len(pts):
            errs.append(match_error(source_locations[gidx, :NUM_SRC], pts))
    return n_ok/len(idx_subset), float(np.nanmean(errs))

_cal = val_idx[:300]
print(f"{'prom_frac':>10} {'correct N':>10} {'mean err':>10}")
results = []
for pf in [0.05, 0.08, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50]:
    rate, err = evaluate_detector(pf, _cal)
    results.append((pf, rate, err))
    print(f"{pf:>10.2f} {rate*100:>9.1f}% {err:>10.4f}")

PROM_FRAC = max(results, key=lambda r: (round(r[1], 3), -r[2]))[0]
print(f"\nselected PROM_FRAC = {PROM_FRAC}")

_r, _e = evaluate_detector(PROM_FRAC, _cal, refine=False)
_rr, _ee = evaluate_detector(PROM_FRAC, _cal, refine=True)
print(f"sub-grid refinement: mean error {_e:.4f} -> {_ee:.4f}")

## Cell 11 — Why do two well-separated true sources look merged in the reconstruction?

The three-panel figure below splits the blur into its causes. Panel 2 is the
**best the trunk can do** — the `nnls` fit from Cell 3, i.e. the output of a
hypothetical perfect branch network. Panel 3 is what the trained network
actually produces. The profile along the line joining the two sources makes the
split quantitative:

* the gap between panel 1 and panel 2 is the **trunk** (fix it with `RBF_H`,
  `RBF_S`, `SIGMA_TGT` — Cell 3's two rules);
* the gap between panel 2 and panel 3 is the **network** (fix it with input
  standardisation, longer training, augmentation, a sharper loss).

With the original settings the trunk alone caps the peak at 0.76 and leaves a
saddle at 0.24 between two lobes 0.30 apart; the trained network reached only
0.59. So both causes were active, and the trunk was the larger one.

In [ ]:
# ============================================================================
# CELL 11: [DIAG] trunk blur vs network blur
# ============================================================================
demo_sources = np.array([[0.0, 0.15], [0.0, -0.15]])   # separation 0.30
if not (D_MIN <= 0.30 <= D_MAX):
    print(f"NOTE: separation 0.30 is OUTSIDE the training range "
          f"[{D_MIN:.4f}, {D_MAX:.4f}] - this is an extrapolation test.\n"
          f"      (The original notebook's demo used 0.30 while d_min was "
          f"{WAVELENGTH/4:.4f}, so it was already extrapolating.)")

I_target      = indicator_target(demo_sources).reshape(N_GRID, N_GRID)
I_trunk_best, _ = best_nonneg_fit(V, indicator_target(demo_sources))
I_trunk_best  = I_trunk_best.reshape(N_GRID, N_GRID)
I_network     = predict_indicator(demo_sources)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, img, ttl in zip(
        axes,
        [I_target, I_trunk_best, I_network],
        [f"true indicator (peak {I_target.max():.2f})",
         f"best possible with this trunk (peak {I_trunk_best.max():.2f})",
         f"trained network (peak {I_network.max():.2f})"]):
    im = ax.imshow(img, extent=[DOMAIN_MIN, DOMAIN_MAX, DOMAIN_MIN, DOMAIN_MAX],
                   origin="lower", aspect="equal", vmin=0, vmax=1,
                   interpolation="nearest")     # no smoothing by the renderer
    ax.scatter(demo_sources[:, 0], demo_sources[:, 1], marker="x", s=80, c="r")
    ax.set_title(ttl, fontsize=9)
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()

col = N_GRID//2
plt.figure(figsize=(6.5, 3.4))
for img, lbl in [(I_target, "true"), (I_trunk_best, "trunk best possible"),
                 (I_network, "network")]:
    plt.plot(GRID, img[:, col], label=f"{lbl} (peak {img[:, col].max():.2f})")
plt.axvline(0.15, color="k", ls=":", lw=1); plt.axvline(-0.15, color="k", ls=":", lw=1)
plt.xlabel("$x_2$ along the line joining the sources"); plt.ylabel("indicator")
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

def valley_depth(img, col=col):
    prof = img[:, col]
    i1, i2 = np.argmin(np.abs(GRID - 0.15)), np.argmin(np.abs(GRID + 0.15))
    lo, hi = min(i1, i2), max(i1, i2)
    saddle, peak = prof[lo:hi+1].min(), max(prof[i1], prof[i2])
    return peak, saddle, 100*(1 - saddle/peak) if peak > 0 else 0.0

for img, lbl in [(I_target, "true"), (I_trunk_best, "trunk best"), (I_network, "network")]:
    pk, sd, dip = valley_depth(img)
    print(f"{lbl:12s} peak={pk:.3f} saddle={sd:.3f} valley depth={dip:5.1f}%")

## Cell 12 — Single held-out sample

In [ ]:
# ============================================================================
# CELL 12: one unseen test sample
# ============================================================================
j = 100
sample_id  = test_idx[j]
true_src   = source_locations[sample_id, :NUM_SRC]
y_true     = Y_test[j].numpy().reshape(N_GRID, N_GRID)
model.eval()
with torch.no_grad():
    y_pred = model(X_test[j:j+1].to(device)).cpu().numpy().reshape(N_GRID, N_GRID)

pred_src, prom = detect_sources(y_pred, min_sep=D_MIN, n_expected=NUM_SRC,
                                prom_frac=PROM_FRAC)

print(f"separation = {np.linalg.norm(true_src[0]-true_src[1]):.4f}")
print("true      :", np.round(true_src, 4).tolist())
print("predicted :", np.round(pred_src, 4).tolist())
print(f"mean localisation error = {match_error(true_src, pred_src):.5f}")
print(f"prediction min/max      = {y_pred.min():.4f} / {y_pred.max():.4f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
for ax, img, ttl in zip(axes, [y_true, y_pred],
                        [r"true $I_{\rm true}(z)$", r"predicted $I_\theta(z)$"]):
    im = ax.imshow(img, extent=[DOMAIN_MIN, DOMAIN_MAX, DOMAIN_MIN, DOMAIN_MAX],
                   origin="lower", aspect="equal", interpolation="nearest")
    ax.scatter(true_src[:, 0], true_src[:, 1], marker="x", s=90, c="r", label="true")
    if len(pred_src):
        ax.scatter(pred_src[:, 0], pred_src[:, 1], marker="o", s=110,
                   facecolors="none", edgecolors="w", label="detected")
    ax.set_title(ttl); ax.legend(fontsize=7)
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()

## Cell 13 — Benchmark: localisation error against separation

The single number worth reporting for this problem is not the training loss but
the localisation error as a function of source separation, since that is what
"resolving sources closer than half a wavelength" actually means.

In [ ]:
# ============================================================================
# CELL 13: [DIAG] error vs separation over many fresh configurations
# ============================================================================
N_BENCH  = 300
bench_rng = np.random.default_rng(12345)
seps, errs, counts = [], [], []

for _ in range(N_BENCH):
    src = sample_pair(bench_rng)
    I   = predict_indicator(src)
    pts, _ = detect_sources(I, min_sep=D_MIN, n_expected=NUM_SRC,
                            prom_frac=PROM_FRAC)
    seps.append(np.linalg.norm(src[0] - src[1]))
    counts.append(len(pts))
    errs.append(match_error(src, pts))

seps, errs, counts = np.array(seps), np.array(errs), np.array(counts)
print(f"correct source count : {100*np.mean(counts == NUM_SRC):.1f}%")
print(f"mean  localisation error : {np.nanmean(errs):.5f}"
      f"  ({np.nanmean(errs)/WAVELENGTH:.4f} wavelengths)")
print(f"median localisation error: {np.nanmedian(errs):.5f}")
print(f"grid spacing for reference: {GRID_H:.5f}")

edges   = np.linspace(seps.min(), seps.max(), 7)
centers = 0.5*(edges[:-1] + edges[1:])
binned  = [np.nanmean(errs[(seps >= a) & (seps < b)]) for a, b in zip(edges[:-1], edges[1:])]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].scatter(seps, errs, s=8, alpha=0.4)
axes[0].plot(centers, binned, "r-o", lw=2, label="bin mean")
axes[0].axhline(GRID_H, color="k", ls="--", lw=1, label="grid spacing")
axes[0].set_xlabel("true separation"); axes[0].set_ylabel("localisation error")
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)
axes[1].hist(errs[~np.isnan(errs)], bins=30)
axes[1].axvline(GRID_H, color="k", ls="--", lw=1)
axes[1].set_xlabel("localisation error"); axes[1].set_ylabel("count")
plt.tight_layout(); plt.show()

### Cell 13b — Old detector vs new, on *your* trained model

The 85.5 % → 95.5 % figure quoted in `RESULTS.md` was measured on synthetic
fields built from the `nnls` trunk fit, because it had to be produced before any
model was trained. This cell re-measures it on the real predictions of the model
you just trained, so you are not taking that number on faith. Run it and compare.

In [ ]:
# ============================================================================
# CELL 13b: [DIAG] reproduce the detector comparison on real model output
# ============================================================================
def detect_sources_original(I, peak_threshold=0.60, neighborhood_size=5):
    '''The original notebook's recipe, kept verbatim for comparison.'''
    local_max = I == maximum_filter(I, size=neighborhood_size)
    strong    = I >= peak_threshold*I.max()
    idx = np.argwhere(local_max & strong)
    if len(idx):
        idx = idx[np.argsort(-I[idx[:, 0], idx[:, 1]])]
    return np.array([[GRID[c], GRID[r]] for r, c in idx]).reshape(-1, 2)

N_CMP = 200
cmp_rng = np.random.default_rng(777)
old_ok, new_ok, old_err, new_err = 0, 0, [], []

for _ in range(N_CMP):
    src = sample_pair(cmp_rng)
    I   = predict_indicator(src)
    p_old = detect_sources_original(I)
    p_new, _ = detect_sources(I, min_sep=D_MIN, n_expected=NUM_SRC,
                              prom_frac=PROM_FRAC)
    old_ok += int(len(p_old) == NUM_SRC)
    new_ok += int(len(p_new) == NUM_SRC)
    if len(p_old): old_err.append(match_error(src, p_old))
    if len(p_new): new_err.append(match_error(src, p_new))

print(f"over {N_CMP} fresh two-source configurations, on the trained model:\n")
print(f"{'detector':<34} {'correct N':>10} {'mean err':>10}")
print(f"{'original (thr 0.60, size 5)':<34} {100*old_ok/N_CMP:>9.1f}% "
      f"{np.mean(old_err):>10.4f}")
print(f"{'prominence + NMS + sub-grid':<34} {100*new_ok/N_CMP:>9.1f}% "
      f"{np.mean(new_err):>10.4f}")

## Cell 14 — A gallery of fresh two-source tests

In [ ]:
# ============================================================================
# CELL 14: gallery
# ============================================================================
num_tests = 12
gal_rng = np.random.default_rng(2024)
ncol = 4
nrow = int(np.ceil(num_tests/ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(3.4*ncol, 3.1*nrow))
axes = np.atleast_1d(axes).ravel()

for t in range(num_tests):
    src = sample_pair(gal_rng)
    I   = predict_indicator(src)
    pts, _ = detect_sources(I, min_sep=D_MIN, n_expected=NUM_SRC,
                            prom_frac=PROM_FRAC)
    sep = np.linalg.norm(src[0] - src[1])
    err = match_error(src, pts)
    ax  = axes[t]
    ax.imshow(I, extent=[DOMAIN_MIN, DOMAIN_MAX, DOMAIN_MIN, DOMAIN_MAX],
              origin="lower", aspect="equal", interpolation="nearest")
    ax.scatter(src[:, 0], src[:, 1], marker="x", s=70, c="r")
    if len(pts):
        ax.scatter(pts[:, 0], pts[:, 1], marker="o", s=90,
                   facecolors="none", edgecolors="w")
    ax.set_title(f"d={sep:.3f}  err={err:.4f}  N={len(pts)}", fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])
for t in range(num_tests, len(axes)):
    axes[t].axis("off")
plt.tight_layout(); plt.show()

## Appendix — what was removed, and other notes

**The 5-fold cross-validation cell is gone.** It cost 5 × 60 = 300 epochs and
produced neither a model nor a decision: the five folds returned
2.108, 2.112, 2.122, 2.128, 2.152 (×10⁻²), a spread of 1.6e-4 on a mean of
2.12e-2. With 48 000 training samples the fold-to-fold variance was never going
to be the thing limiting you, and `best_fold_val = min(...)` over epochs is in
any case an optimistically-biased estimate (the minimum of a noisy sequence).
If you want to spend that compute, spend it sweeping `RBF_H`, `LOSS_W` or
`W_DICE` — those change the answer; the fold index does not.

**Memory.** `I_true` at 60 000 × 4 096 float32 is 983 MB, plus 246 MB for
`X_all`. If you hit a Colab RAM limit, drop `Y_train` entirely and build the
training targets on the fly with `targets_from_sources` (Cell 7) — with
`ROT_AUGMENT = True` they are already rebuilt every batch, so `Y_train` is only
read when augmentation is off. Keep `Y_val` and `Y_test`.

**`weight_decay` on `Adam` was not doing what it looks like.** `torch.optim.Adam`
adds L2 to the gradient, which the adaptive per-parameter scaling then rescales;
`AdamW` applies the decoupled decay that `weight_decay` is normally understood to
mean. Switched.

**The demo separation was out of distribution.** The original "completely new
test data" cell used sources at `(0, 0)` and `(0, −0.30)`, a separation of 0.30,
while the training set only ever contained separations ≥ `λ/4 = 0.3142`. That
figure was an extrapolation. If you want to test at 0.30, train with
`D_MIN = 0.30` or lower.

**Duplicate-pair detection.** `used_signatures` stores 60 000 coordinate tuples
so an `assert` can confirm no pair repeats. With continuous uniform sampling the
collision probability is zero, so the assert cannot fail; it is harmless but it
is not buying anything.